# CompressKit — end-to-end on Kaggle (one notebook, **Run All**)

`train FP32 baseline → gradual prune + fine-tune → quantize (always last) → evaluate the exported .onnx → report`

**Before running:** right panel → *Accelerator: GPU T4*, *Internet: ON*. For the HAR model, *Add Input* →
*Human Action Recognition (HAR) Dataset*. Then edit only the config cell below.

Outputs land in `/kaggle/working/compresskit_output/`: the final `model.onnx`, `report.html`, `results.json`.
**Latency in this run is Kaggle-CPU latency**, not your deployment target — measure that with
`compresskit bench model.onnx` on the target machine (see the last cell).

In [ ]:
# ======================= CONFIG — the only cell you edit =======================
MODEL      = "har_resnet18"   # "har_resnet18" | "resnet20" | "resnet56" | "custom"
MODE       = "manual"         # "manual": you choose the schedule | "auto": give a budget, it finds the least compression

# --- manual mode ---
PRUNE_RATIO = 0.5             # fraction of channels to remove
STEPS       = 5               # gradual pruning steps
FT_EPOCHS   = 5               # fine-tune epochs after each step
LOSS        = "dkd"           # "kd" | "dkd"

# --- auto mode (accuracy/size budgets are valid on Kaggle; a latency budget would be Kaggle-CPU latency) ---
BUDGET      = "accuracy_drop<1%,size<5mb"

PRECISION   = "int8"          # Kaggle's CPU has no fast INT8, but we still build the INT8 model for your target
TRAIN_EPOCHS = None           # None = default (60 for CIFAR models, 10 for pretrained HAR/custom)

# --- bring your own data / model (MODEL = "custom") ---
DATA_DIR    = "/kaggle/input/my-dataset"   # one sub-folder per class
INPUT_SIZE  = 224
MODEL_FILE  = None            # path to a .pt saved with compresskit's save_model; None = train a pretrained ResNet-18
# ===============================================================================

In [ ]:
import os, subprocess, sys, shutil, json, pathlib
REPO = "https://github.com/Bhagyesh0603/Model-Compression-ToolKit..git"
if not os.path.exists("/kaggle/working/compresskit_repo"):
    subprocess.run(["git", "clone", "-q", REPO, "/kaggle/working/compresskit_repo"], check=True)
os.chdir("/kaggle/working/compresskit_repo")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)

import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE — enable GPU in the right panel, then Restart & Run All")

In [ ]:
ARCH, DATASET, DATA_ROOT = {
    "har_resnet18": ("har_resnet18", "har", None),
    "resnet20":     ("resnet20", "cifar10", None),
    "resnet56":     ("resnet56", "cifar10", None),
    "custom":       ("har_resnet18", "folder", DATA_DIR),
}[MODEL]

OUT  = pathlib.Path("/kaggle/working/compresskit_output"); OUT.mkdir(parents=True, exist_ok=True)
WORK = "/kaggle/working/work"
CKPT = MODEL_FILE or f"checkpoints/{MODEL}_fp32.pt"

def run(args):
    args = [str(a) for a in args]
    print("$ compresskit", " ".join(args), flush=True)
    code = subprocess.run([sys.executable, "-m", "compresskit.cli", *args]).returncode
    return code

data_args = ["--dataset", DATASET, "--input-size", INPUT_SIZE] + (["--data-root", DATA_ROOT] if DATA_ROOT else [])

## 1 · Train the FP32 baseline (skipped if the checkpoint already exists)

In [ ]:
if MODEL_FILE is None:
    train_args = ["train", "--arch", ARCH, "--out", CKPT, "--skip-if-exists", *data_args]
    if TRAIN_EPOCHS: train_args += ["--epochs", TRAIN_EPOCHS]
    assert run(train_args) == 0, "training failed"
else:
    print("using provided model:", MODEL_FILE)

## 2 · Compress: gradual prune + fine-tune → quantize → evaluate the .onnx

In [ ]:
common = ["--model", CKPT, *data_args, "--precision", PRECISION, "--allow-slow-int8", "--workdir", WORK,
          "--report", OUT / "report.html", "--json", OUT / "results.json", "--loss", LOSS]
if MODE == "manual":
    code = run(["pipeline", *common, "--prune", PRUNE_RATIO, "--steps", STEPS, "--fine-tune-epochs", FT_EPOCHS])
else:
    code = run(["compress", *common, "--budget", BUDGET, "--steps", STEPS, "--fine-tune-epochs", FT_EPOCHS])
print("exit code", code, "(auto mode returns 1 if the budget was not met — the best attempt is still saved)")

## 3 · Collect the outputs

In [ ]:
results = json.loads((OUT / "results.json").read_text())
final = results["stages"][-1]
shutil.copy(final["artifact"], OUT / "model.onnx")
base = results["baseline"]
print(f"accuracy : {base['accuracy']*100:.2f}% -> {final['accuracy']*100:.2f}%  (drop {results['totals']['accuracy_drop_pp']:.2f} pp)")
print(f"size     : {base['size_mb']:.2f} MB -> {final['size_mb']:.2f} MB  ({results['totals']['size_reduction']:.2f}x smaller)")
print(f"precision: {final['precision']}")
print("(latency in this run was measured on Kaggle's CPU — not your target)")
for f in sorted(OUT.iterdir()): print(f"{f.stat().st_size/1e6:8.2f} MB  {f.name}")
from IPython.display import IFrame, display
display(IFrame(str(OUT / "report.html"), width="100%", height=900))

## 4 · Verify speed on your real target (once, on your own machine)

Download `compresskit_output/` from the *Output* tab, then on the target machine:

```bash
pip install -e .   # the repo
compresskit bench model.onnx --input-shape 1 3 224 224      # 1 3 32 32 for the CIFAR models
```